# LoRA and QLoRA Demo — Kubeflow Edition

This is the Kubeflow-friendly version of the LoRA / QLoRA demo (originally written for Google Colab).

**What this notebook does**
- Verifies the Kubeflow environment is ready.
- Creates a very small toy instruction dataset.
- Loads a small base model (TinyLlama 1.1B).
- Fine-tunes once with **LoRA**.
- Fine-tunes once with **QLoRA** (4-bit base).
- Runs a quick inference comparison.
- Saves both adapters under your home directory (PVC-backed, survives pod restarts).
- Benchmarks Full FT vs LoRA vs QLoRA on a single step.

**Prerequisites (one-time setup per pod)**

Before running this notebook, in a terminal:

```bash
bash ~/setup_env.sh        # installs pinned course baseline
# then: Kernel  ->  Restart Kernel
python ~/check_env.py      # should show all checks passing
```

If `~/setup_env.sh` is not in your home directory, see the student setup guide.

**About bitsandbytes**

A library for running and fine-tuning large neural networks with less GPU memory using low-bit (k-bit) representations and optimized GPU kernels. Used here for QLoRA 4-bit base quantization.

## 1. Environment check (Kubeflow)

This cell **fails fast** if the pod is not properly set up. If anything fails, run `bash ~/setup_env.sh` in a terminal and restart the kernel.

In [1]:
import sys, importlib

REQUIRED = ["torch", "transformers", "datasets", "peft", "accelerate", "bitsandbytes"]
missing = []
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    raise RuntimeError(
        f"Missing packages: {missing}. "
        "Run `bash ~/setup_env.sh` in a terminal, then Restart Kernel."
    )

import torch
print(f"Python       : {sys.version.split()[0]}")
print(f"Torch        : {torch.__version__} (CUDA build {torch.version.cuda})")
print(f"CUDA visible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory   : {total_gb:.1f} GB")
else:
    print("WARNING: No CUDA GPU detected. QLoRA section will not run.")
    print("Spawn a GPU pod (e.g., jupyter-pytorch-cuda-full + 1 GPU).")

RuntimeError: Missing packages: ['torch', 'transformers', 'datasets', 'peft', 'accelerate', 'bitsandbytes']. Run `bash ~/setup_env.sh` in a terminal, then Restart Kernel.

## 1.5. Model picker

Pick a model by changing `MODEL_KEY` below. The cell estimates whether the model fits with LoRA or only with QLoRA on your current GPU, and suggests training hyperparameters that are applied automatically in later cells.

**Quick guide for an A100 80GB:**
- ≤3B: full-throttle, fast iteration (LoRA fits easily, batch size 2+)
- 7–9B: best LoRA sweet spot
- 14–34B: LoRA tight, QLoRA easy
- 70B+: QLoRA only

**License note:** Models marked "gated" (Llama, Gemma) require accepting the license on Hugging Face and running `huggingface-cli login` once. Qwen models are ungated — easiest for students.

In [2]:
# ============================================================
# MODEL PICKER  --  change MODEL_KEY below to switch models
# ============================================================
MODEL_KEY = "tinyllama"   # <-- pick from the registry below

MODEL_REGISTRY = {
    # key                : (hf_id,                                  params_B, gated)
    # ---- Small (<=4B): fast iteration ----
    "tinyllama":           ("TinyLlama/TinyLlama-1.1B-Chat-v1.0",      1.1, False),
    "qwen2.5-1.5b":        ("Qwen/Qwen2.5-1.5B-Instruct",              1.5, False),
    "qwen2.5-3b":          ("Qwen/Qwen2.5-3B-Instruct",                3.0, False),
    "llama3.2-1b":         ("meta-llama/Llama-3.2-1B-Instruct",        1.2, True),
    "llama3.2-3b":         ("meta-llama/Llama-3.2-3B-Instruct",        3.2, True),
    "gemma2-2b":           ("google/gemma-2-2b-it",                    2.6, True),
    "phi3-mini":           ("microsoft/Phi-3-mini-4k-instruct",        3.8, False),
    # ---- Medium (7-14B): great LoRA fit on A100 80GB ----
    "qwen2.5-7b":          ("Qwen/Qwen2.5-7B-Instruct",                7.6, False),
    "llama3.1-8b":         ("meta-llama/Llama-3.1-8B-Instruct",        8.0, True),
    "mistral-7b-v0.3":     ("mistralai/Mistral-7B-Instruct-v0.3",      7.2, False),
    "gemma2-9b":           ("google/gemma-2-9b-it",                    9.2, True),
    "qwen2.5-14b":         ("Qwen/Qwen2.5-14B-Instruct",              14.7, False),
    # ---- Large (27-34B): LoRA tight, QLoRA easy ----
    "gemma2-27b":          ("google/gemma-2-27b-it",                  27.2, True),
    "qwen2.5-32b":         ("Qwen/Qwen2.5-32B-Instruct",              32.5, False),
    "yi1.5-34b":           ("01-ai/Yi-1.5-34B-Chat",                  34.4, False),
    # ---- Very large (70B+): QLoRA only ----
    "llama3.1-70b":        ("meta-llama/Llama-3.1-70B-Instruct",      70.6, True),
    "llama3.3-70b":        ("meta-llama/Llama-3.3-70B-Instruct",      70.6, True),
    "qwen2.5-72b":         ("Qwen/Qwen2.5-72B-Instruct",              72.7, False),
}

assert MODEL_KEY in MODEL_REGISTRY, (
    f"Unknown MODEL_KEY '{MODEL_KEY}'. Choose from:\n  " + "\n  ".join(MODEL_REGISTRY)
)
MODEL_ID, params_b, gated = MODEL_REGISTRY[MODEL_KEY]

# --- Rough memory estimates (GB) ---
fp16_weights_gb  = params_b * 2.0
int4_weights_gb  = params_b * 0.5
lora_train_gb    = fp16_weights_gb * 2.0
qlora_train_gb   = int4_weights_gb + params_b * 0.2

# --- GPU info ---
import torch
gpu_total_gb = (
    torch.cuda.get_device_properties(0).total_memory / 1e9
    if torch.cuda.is_available() else 0.0
)
budget = gpu_total_gb * 0.9
lora_fits  = lora_train_gb  <= budget
qlora_fits = qlora_train_gb <= budget

# --- Suggested training args by size ---
if   params_b <= 3:   rec = dict(max_length=128, batch_size=2, accum=1)
elif params_b <= 9:   rec = dict(max_length=128, batch_size=1, accum=2)
elif params_b <= 14:  rec = dict(max_length=96,  batch_size=1, accum=4)
elif params_b <= 34:  rec = dict(max_length=64,  batch_size=1, accum=8)
else:                 rec = dict(max_length=64,  batch_size=1, accum=16)

SUGGESTED_MAX_LENGTH = rec["max_length"]
SUGGESTED_BATCH_SIZE = rec["batch_size"]
SUGGESTED_ACCUM      = rec["accum"]

print("=" * 60)
print(f" Selected model      : {MODEL_KEY}")
print(f" HF model ID         : {MODEL_ID}")
print(f" Parameters          : ~{params_b} B")
print(f" License gated       : {'YES - huggingface-cli login + accept license' if gated else 'no'}")
print("=" * 60)
print(f" GPU memory          : {gpu_total_gb:.1f} GB")
print(f" Est. fp16 weights   : ~{fp16_weights_gb:.1f} GB")
print(f" Est. 4-bit weights  : ~{int4_weights_gb:.1f} GB")
print("-" * 60)
print(f" Fits with LoRA?     : {'YES' if lora_fits else 'NO  -> use QLoRA'}"
      f"   (peak ~{lora_train_gb:.1f} GB)")
print(f" Fits with QLoRA?    : {'YES' if qlora_fits else 'NO  -> model too large for this GPU'}"
      f"   (peak ~{qlora_train_gb:.1f} GB)")
print("-" * 60)
print(" Suggested training args for this model size:")
print(f"   max_length                  = {SUGGESTED_MAX_LENGTH}")
print(f"   per_device_train_batch_size = {SUGGESTED_BATCH_SIZE}")
print(f"   gradient_accumulation_steps = {SUGGESTED_ACCUM}")
print("=" * 60)

if gated:
    print()
    print("Reminder: gated model. If you have not already, run in a terminal:")
    print("   huggingface-cli login")
    print("   # then accept the license on the model's HF page in your browser")

 Selected model      : tinyllama
 HF model ID         : TinyLlama/TinyLlama-1.1B-Chat-v1.0
 Parameters          : ~1.1 B
 License gated       : no
 GPU memory          : 85.0 GB
 Est. fp16 weights   : ~2.2 GB
 Est. 4-bit weights  : ~0.6 GB
------------------------------------------------------------
 Fits with LoRA?     : YES   (peak ~4.4 GB)
 Fits with QLoRA?    : YES   (peak ~0.8 GB)
------------------------------------------------------------
 Suggested training args for this model size:
   max_length                  = 128
   per_device_train_batch_size = 2
   gradient_accumulation_steps = 1


## 2. Imports and setup

Imports the Python packages used throughout the notebook and sets up a working directory under your home folder. Anything saved under `~/lora_qlora_demo/` lives on the PVC and survives pod restarts.

In [3]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,                          # Ready-made training loop.
    TrainingArguments,                # Hyperparameters and I/O config.
    DataCollatorForLanguageModeling,  # Batching helper for language modeling.
    BitsAndBytesConfig,               # bitsandbytes quantization config.
)
from peft import (
    LoraConfig,
    get_peft_model,                   # Wraps a base model with PEFT adapters.
    prepare_model_for_kbit_training,  # Helper for training with quantized backbones.
)

# --- Kubeflow working directory (PVC-backed, survives pod restarts) ---
WORK_DIR          = os.path.expanduser("~/lora_qlora_demo")
LORA_OUT_DIR      = f"{WORK_DIR}/lora_demo_out"
LORA_ADAPTER_DIR  = f"{WORK_DIR}/lora_adapter"
QLORA_OUT_DIR     = f"{WORK_DIR}/qlora_demo_out"
QLORA_ADAPTER_DIR = f"{WORK_DIR}/qlora_adapter"
os.makedirs(WORK_DIR, exist_ok=True)
print(f"Outputs will go under: {WORK_DIR}")

# Put the Hugging Face cache on the PVC too so downloaded models persist across restarts
HF_CACHE = os.path.expanduser("~/.cache/huggingface")
os.environ.setdefault("HF_HOME", HF_CACHE)
os.makedirs(HF_CACHE, exist_ok=True)
print(f"HF cache             : {HF_CACHE}")


def print_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    pct = 100 * trainable / total
    print(f"Trainable params: {trainable:,}")
    print(f"Total params: {total:,}")
    print(f"Trainable %: {pct:.4f}%")

Outputs will go under: /home/jovyan/lora_qlora_demo
HF cache             : /home/jovyan/.cache/huggingface


## 3. Confirm we have a GPU

LoRA can run on CPU (slowly). QLoRA requires a CUDA GPU.

In [4]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The LoRA section will still run on CPU but slowly;")
    print("the QLoRA section will raise an error. Spawn a GPU pod to run end-to-end.")

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


## 4. Create a tiny demo dataset

Each example is converted to a single text string with an instruction and a response.

In [5]:
examples = [
    {"instruction": "What is LoRA?",
     "response": "LoRA fine-tunes a model by training small low-rank adapter matrices instead of updating all model weights."},
    {"instruction": "What is QLoRA?",
     "response": "QLoRA combines 4-bit quantized base weights with LoRA adapters so training uses much less GPU memory."},
    {"instruction": "Why is LoRA memory efficient?",
     "response": "LoRA trains only a small set of added adapter parameters, so memory use is much lower than full fine-tuning."},
    {"instruction": "Why is QLoRA even more memory efficient?",
     "response": "QLoRA stores the base model in 4-bit form and trains only LoRA adapters, which reduces memory further."},
    {"instruction": "What does rank mean in LoRA?",
     "response": "Rank controls the size of the low-rank update matrices, so it affects adapter capacity and memory cost."},
    {"instruction": "What is gradient checkpointing?",
     "response": "Gradient checkpointing saves memory by recomputing some activations during backpropagation instead of storing all of them."},
]

formatted = []
for ex in examples:
    text = f"Instruction:\n{ex['instruction']}\n\nResponse:\n{ex['response']}"
    formatted.append({"text": text})

dataset = Dataset.from_list(formatted)
dataset

Dataset({
    features: ['text'],
    num_rows: 6
})

## 5. Load tokenizer

If the tokenizer doesn't have a padding token, reuse the EOS token so batching works cleanly.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## 6. Tokenize the dataset

Keep sequence length small so the notebook stays lightweight.

In [7]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=SUGGESTED_MAX_LENGTH,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
tokenized_dataset

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 6
})

## 7. Pick LoRA target modules

LoRA inserts trainable adapters into selected linear layers. This helper finds common transformer projection layers (attention + MLP) by their suffix names like `q_proj`, `k_proj`, etc.

In [8]:
def pick_lora_targets(model):
    common_names = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    found = set()
    for name, module in model.named_modules():
        last = name.split(".")[-1]
        if last in common_names:
            found.add(last)
    return sorted(found)

## 8. Load the base model for LoRA

Loads the base model in standard precision and adds LoRA adapters on the chosen target modules.

- Base model weights stay **frozen**.
- Only the **adapter** weights are trainable.

In [9]:
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
base_model.config.use_cache = False

# Move to GPU if available for faster LoRA training
if torch.cuda.is_available():
    base_model = base_model.to("cuda")

lora_targets = pick_lora_targets(base_model)
print("LoRA target modules:", lora_targets)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=lora_targets,
)

lora_model = get_peft_model(base_model, lora_config)
print_trainable_parameters(lora_model)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

LoRA target modules: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
Trainable params: 6,307,840
Total params: 1,106,356,224
Trainable %: 0.5701%


| Suffix | Block | Role | What changes if you LoRA-tune it |
|--------|-------|------|----------------------------------|
| q_proj | Self-attention | Produces queries (Q) | Changes what each token asks for when attending |
| k_proj | Self-attention | Produces keys (K) | Changes what information each token offers |
| v_proj | Self-attention | Produces values (V) | Changes the content that flows through attention |
| o_proj | Self-attention | Output projection after heads merge | Changes how attended info is mixed back in |
| gate_proj | MLP (SwiGLU-style) | Gate projection | Changes which MLP features turn on/off |
| up_proj | MLP | Hidden → intermediate | Changes feature expansion in the MLP |
| down_proj | MLP | Intermediate → hidden | Changes how expanded features are compressed back |
| lm_head | Output head | Hidden → vocab logits | Directly changes token logits |

## 9. Train the LoRA model

Outputs go under your PVC-backed home directory.

In [10]:
lora_args = TrainingArguments(
    output_dir=LORA_OUT_DIR,
    per_device_train_batch_size=SUGGESTED_BATCH_SIZE,
    gradient_accumulation_steps=SUGGESTED_ACCUM,
    max_steps=20,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer_lora = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer_lora.train()
lora_model.save_pretrained(LORA_ADAPTER_DIR)
tokenizer.save_pretrained(LORA_ADAPTER_DIR)
print(f"Saved LoRA adapter to {LORA_ADAPTER_DIR}")

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
1,4.189500
2,3.912600
3,3.423700
4,3.677800
5,2.678600
6,3.575900
7,2.731100
8,3.138000
9,1.866500
10,1.871400


Saved LoRA adapter to /home/jovyan/lora_qlora_demo/lora_adapter


## 10. Why QLoRA is different

- **LoRA:** base model is loaded normally, adapters are trained.
- **QLoRA:** base model is loaded in **4-bit** form, then adapters are trained on top.

QLoRA usually reduces GPU memory a lot, which is why it's popular in limited-GPU settings.

## 11. Load the base model in 4-bit for QLoRA

Sets up 4-bit quantization with `bitsandbytes`, enables gradient checkpointing, and prepares the quantized model for k-bit training.

In [11]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "QLoRA requires a CUDA GPU. Spawn a GPU notebook pod in Kubeflow "
        "(e.g., jupyter-pytorch-cuda-full image with 1 GPU attached)."
    )

major_cc, _ = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if major_cc >= 8 else torch.float16
print("Using compute dtype:", compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

base_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
base_4bit.config.use_cache = False
base_4bit.gradient_checkpointing_enable()
base_4bit = prepare_model_for_kbit_training(base_4bit)

qlora_targets = pick_lora_targets(base_4bit)
print("QLoRA target modules:", qlora_targets)

qlora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=qlora_targets,
)

qlora_model = get_peft_model(base_4bit, qlora_config)
print_trainable_parameters(qlora_model)

Using compute dtype: torch.bfloat16
QLoRA target modules: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
Trainable params: 6,307,840
Total params: 621,914,112
Trainable %: 1.0143%


## 12. Train the QLoRA model

Uses the paged 8-bit AdamW optimizer to keep optimizer state memory low.

In [12]:
qlora_args = TrainingArguments(
    output_dir=QLORA_OUT_DIR,
    per_device_train_batch_size=SUGGESTED_BATCH_SIZE,
    gradient_accumulation_steps=max(SUGGESTED_ACCUM, 2),
    max_steps=10,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer_qlora = Trainer(
    model=qlora_model,
    args=qlora_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer_qlora.train()
qlora_model.save_pretrained(QLORA_ADAPTER_DIR)
tokenizer.save_pretrained(QLORA_ADAPTER_DIR)
print(f"Saved QLoRA adapter to {QLORA_ADAPTER_DIR}")

max_steps is given, it will override any value given in num_train_epochs
/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,4.041800
2,4.839600
3,5.191300
4,3.241800
5,3.759200
6,4.339300
7,2.714600
8,3.243100
9,3.434000
10,2.363600


Saved QLoRA adapter to /home/jovyan/lora_qlora_demo/qlora_adapter


## 13. Simple inference helper

In [13]:
@torch.inference_mode()
def generate_text(model, prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## 14. Test both models

Compares LoRA and QLoRA adapters on one tiny prompt. Outputs may still be weak or repetitive because the dataset is extremely small.

In [14]:
prompt = "Instruction:\nExplain QLoRA in 2 short sentences.\n\nResponse:\n"

print("===== LoRA output =====")
print(generate_text(lora_model, prompt))

print("\n===== QLoRA output =====")
print(generate_text(qlora_model, prompt))

===== LoRA output =====


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Instruction:
Explain QLoRA in 2 short sentences.

Response:
QLoRA uses a smaller model, which speeds up inference times and reduces memory usage.

===== QLoRA output =====


/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Instruction:
Explain QLoRA in 2 short sentences.

Response:
QLoRA is a quantum-optimized randomized algorithm that can be used for quantum-resistant encryption, such as RSA. It has been designed to operate at the 1/20000th level of noise, which is orders of magnitude smaller than classical encryption. Q


## 15. Inspect the saved adapters

In Colab, people often save adapters to Google Drive. In Kubeflow, your home directory is already on a persistent volume (PVC), so anything saved under `~/lora_qlora_demo/` survives pod restarts — no Drive mount needed.

In [15]:
import subprocess

print("=== Files saved under WORK_DIR ===")
subprocess.run(["ls", "-lh", WORK_DIR])

for d in [LORA_ADAPTER_DIR, QLORA_ADAPTER_DIR]:
    print(f"\n--- Contents of {d} ---")
    subprocess.run(["ls", "-lh", d])

print("\n=== Total size ===")
subprocess.run(["du", "-sh", WORK_DIR])

=== Files saved under WORK_DIR ===
total 16K
drwxr-sr-x 2 jovyan users 4.0K May 18 14:56 lora_adapter
drwxr-sr-x 2 jovyan users 4.0K May 18 14:56 lora_demo_out
drwxr-sr-x 2 jovyan users 4.0K May 18 14:59 qlora_adapter
drwxr-sr-x 2 jovyan users 4.0K May 18 14:59 qlora_demo_out

--- Contents of /home/jovyan/lora_qlora_demo/lora_adapter ---
total 29M
-rw-r--r-- 1 jovyan users  736 May 18 14:56 adapter_config.json
-rw-r--r-- 1 jovyan users  25M May 18 14:56 adapter_model.safetensors
-rw-r--r-- 1 jovyan users 5.0K May 18 14:56 README.md
-rw-r--r-- 1 jovyan users  551 May 18 14:56 special_tokens_map.json
-rw-r--r-- 1 jovyan users 1.4K May 18 14:56 tokenizer_config.json
-rw-r--r-- 1 jovyan users 3.5M May 18 14:56 tokenizer.json
-rw-r--r-- 1 jovyan users 489K May 18 14:56 tokenizer.model

--- Contents of /home/jovyan/lora_qlora_demo/qlora_adapter ---
total 29M
-rw-r--r-- 1 jovyan users  736 May 18 14:59 adapter_config.json
-rw-r--r-- 1 jovyan users  25M May 18 14:59 adapter_model.safetensors
-

CompletedProcess(args=['du', '-sh', '/home/jovyan/lora_qlora_demo'], returncode=0)

**Sharing adapters off the pod**

- **Hugging Face Hub** — `huggingface-cli login`, then `model.push_to_hub("user/repo-name")`.
- **S3 / GCS** — `aws s3 cp --recursive ...` or `gsutil cp -r ...` from a terminal.
- **Download via JupyterLab** — zip first, then right-click → Download:

```bash
tar czf ~/lora_adapter.tar.gz -C ~/lora_qlora_demo lora_adapter
```

## 16. Compare Full FT vs LoRA vs QLoRA on one step

Builds a full-precision fine-tunable copy of the same model (for comparison only), then measures trainable params, approximate parameter memory, and one-step wall-clock time for each method.

In [ ]:
import time
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_

# Reuses: MODEL_ID, tokenizer, tokenized_dataset, data_collator, lora_model, qlora_model

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def approximate_param_memory(num_params, dtype=torch.float16):
    """Approximate memory in MB used by trainable parameters."""
    bytes_per_param = torch.finfo(dtype).bits // 8
    return num_params * bytes_per_param / (1024**2)

def make_dummy_batch(tokenizer, dataset, collator, device, seq_len=64, batch_size=1):
    """Use first batch from existing dataset; fall back to synthetic if needed."""
    if dataset is not None and len(dataset) > 0:
        dl = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)
        batch = next(iter(dl))
    else:
        inputs = tokenizer(
            ["Dummy training example."] * batch_size,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=seq_len,
        )
        batch = inputs
    return {k: v.to(device) for k, v in batch.items()}

def run_one_step(model, optimizer, batch, label_key="labels"):
    """Single forward + backward + optimizer step, timed."""
    model.train()
    for p in model.parameters():
        if p.grad is not None:
            p.grad = None

    if label_key not in batch:
        batch[label_key] = batch["input_ids"].clone()

    start = time.time()
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    end = time.time()
    return loss.item(), end - start

# 1. Full fine-tuning model (no LoRA, no quantization)
full_ft_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
full_ft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=full_ft_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
for p in full_ft_model.parameters():
    p.requires_grad = True

# 2. Counts + memory
full_trainable,  full_total  = count_trainable_params(full_ft_model)
lora_trainable,  lora_total  = count_trainable_params(lora_model)
qlora_trainable, qlora_total = count_trainable_params(qlora_model)

full_mem_mb  = approximate_param_memory(full_trainable,  dtype=full_ft_dtype)
lora_mem_mb  = approximate_param_memory(lora_trainable,  dtype=torch.float16)
qlora_mem_mb = approximate_param_memory(qlora_trainable, dtype=torch.float16)

print("=== Trainable parameters ===")
print(f"Full fine-tuning : {full_trainable:,} / {full_total:,} ({100.0*full_trainable/full_total:.2f}%)")
print(f"LoRA             : {lora_trainable:,} / {lora_total:,} ({100.0*lora_trainable/lora_total:.2f}%)")
print(f"QLoRA            : {qlora_trainable:,} / {qlora_total:,} ({100.0*qlora_trainable/qlora_total:.2f}%)")

print("\n=== Approx. memory for trainable params (MB) ===")
print(f"Full fine-tuning (fp16): ~{full_mem_mb:,.1f} MB")
print(f"LoRA adapters (fp16)   : ~{lora_mem_mb:,.1f} MB")
print(f"QLoRA adapters (fp16)  : ~{qlora_mem_mb:,.1f} MB")
print("\nNote: QLoRA also keeps the base model in 4-bit, so total GPU memory is even lower than this trainable-params view.")

# 3. Dummy batch on each model's device
batch_full  = make_dummy_batch(tokenizer, tokenized_dataset, data_collator, device=next(full_ft_model.parameters()).device)
batch_lora  = make_dummy_batch(tokenizer, tokenized_dataset, data_collator, device=next(lora_model.parameters()).device)
batch_qlora = make_dummy_batch(tokenizer, tokenized_dataset, data_collator, device=next(qlora_model.parameters()).device)

# 4. Optimizers
full_optimizer  = torch.optim.AdamW(full_ft_model.parameters(), lr=2e-4)
lora_optimizer  = torch.optim.AdamW([p for p in lora_model.parameters()  if p.requires_grad], lr=2e-4)
qlora_optimizer = torch.optim.AdamW([p for p in qlora_model.parameters() if p.requires_grad], lr=2e-4)

# 5. One dummy training step timings
print("\n=== One dummy training step (seconds) ===")
full_loss,  full_time  = run_one_step(full_ft_model, full_optimizer,  batch_full)
print(f"Full fine-tuning : loss={full_loss:.4f}, step_time={full_time:.3f} s")

lora_loss,  lora_time  = run_one_step(lora_model,    lora_optimizer,  batch_lora)
print(f"LoRA             : loss={lora_loss:.4f}, step_time={lora_time:.3f} s")

qlora_loss, qlora_time = run_one_step(qlora_model,   qlora_optimizer, batch_qlora)
print(f"QLoRA            : loss={qlora_loss:.4f}, step_time={qlora_time:.3f} s")

print("\nThese numbers come from a single small batch, but they clearly show how LoRA/QLoRA reduce trainable params, memory, and per-step compute vs full fine-tuning.")

## 17. GPU memory snapshot (Kubeflow extra)

A quick GPU memory readout from inside the notebook. Run this before and after loading models to see how much each method actually uses.

In [ ]:
if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info(0)
    used_gb = (total_b - free_b) / 1e9
    total_gb = total_b / 1e9
    alloc_gb = torch.cuda.memory_allocated() / 1e9
    reserved_gb = torch.cuda.memory_reserved() / 1e9
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"Used / Total   : {used_gb:.2f} GB / {total_gb:.2f} GB")
    print(f"Torch allocated: {alloc_gb:.2f} GB")
    print(f"Torch reserved : {reserved_gb:.2f} GB")
else:
    print("No CUDA GPU available.")

## 18. Reflection questions: LoRA vs QLoRA, loss vs memory

In our experiment, LoRA and QLoRA both train low-rank adapters on top of a frozen base model, but LoRA keeps the base weights in fp16/bf16, while QLoRA stores the base in 4-bit quantized form to save VRAM.

**Question 1.** In your own words, explain why QLoRA can sometimes end up with *slightly higher (worse) loss* than LoRA, even though both methods train a similar fraction of parameters. In your answer, explicitly mention:

- How **quantization noise** in the 4-bit base model affects training and final loss.
- Why QLoRA is still attractive in practice because of its **VRAM savings**, especially for larger models that LoRA / full fine-tuning cannot fit.

**Answer.** QLoRA can end up with slightly higher loss than LoRA because it fine-tunes adapters on top of a **4-bit quantized base model**, whereas LoRA keeps the base in higher-precision fp16/bf16. Quantizing the base to 4-bit introduces **quantization noise**: weights and activations are approximated with fewer bits, so they cannot represent the same fine-grained values as the higher-precision model. During training, gradients are computed on these approximated weights, making the optimization landscape noisier and slightly degrading how well the model fits the data — which can lead to a higher final loss compared with LoRA.

Despite this, QLoRA is very attractive in practice because it dramatically **reduces VRAM usage**: the base model is stored in compact 4-bit form, and only a small set of adapter parameters is trained in higher precision. This allows fine-tuning of much larger models (or larger batch sizes) on the same GPU where full fine-tuning or even LoRA with an fp16 base would not fit. QLoRA trades a bit of potential quality (slightly higher loss) for a big gain in memory efficiency, which is often the critical constraint in real-world setups.

---

**Question 2.** Suppose you have access to (a) a 24 GB GPU and (b) an 8 GB GPU. For each case, which fine-tuning method (full fine-tuning, LoRA, or QLoRA) would you choose for a 7B–13B model, and why?

**Answer.** On a 24 GB GPU, a 7B–13B model in fp16 often fits comfortably enough to run **LoRA** with a full-precision base (and sometimes even full fine-tuning if the model is on the smaller end and batch sizes are modest). LoRA is usually preferred here because it keeps the base in higher precision (less quantization noise) while still reducing trainable parameters and memory for optimizer states.

On an 8 GB GPU, the same 7B–13B model in fp16 may not fit at all, or may leave too little room for activations and optimizer states. In that case, **QLoRA** becomes the only practical option: storing the base in 4-bit and training only small adapters in higher precision dramatically reduces VRAM usage so training is feasible at all.

> On a fixed GPU, QLoRA may let you move up one or two model sizes; the gain from a bigger base often outweighs the small quality loss from quantization.

---

### Scenario: LoRA on 7B vs QLoRA on 13B

You have a single 12 GB GPU. You want to fine-tune a model for a domain-specific instruction-following task.

- A 7B model in fp16 fits with some room to spare for LoRA adapters and optimizer states.
- A 13B model in fp16 does **not** fit, but it **does** fit if you load it in 4-bit and use QLoRA adapters.

**Question 3.** Which setup would you choose, and why?

- (A) LoRA on the 7B model (fp16 base + LoRA adapters)
- (B) QLoRA on the 13B model (4-bit base + QLoRA adapters)

Discuss: (1) model capacity vs quantization noise, (2) VRAM constraints.

**Answer.** I would choose **(B) QLoRA on the 13B model**. Even though QLoRA introduces quantization noise from storing the base in 4-bit, the 13B model has significantly higher capacity than the 7B model. On a 12 GB GPU, LoRA with an fp16 base is limited to the 7B model; the 13B in fp16 simply does not fit, so full fine-tuning or LoRA on 13B is infeasible. With QLoRA, the 13B base compresses into 4-bit, fits in the VRAM budget, and small high-precision adapters can still be trained. The gain from the larger model's capacity often outweighs the small quality loss from quantization, so QLoRA on 13B can deliver better downstream performance than LoRA on 7B on the same GPU.

## 19. Takeaways

- **LoRA** reduces trainable parameters by learning small adapter matrices on top of a frozen base.
- **QLoRA** goes further by loading the base model in **4-bit** form, dramatically cutting VRAM.
- Both methods reduce memory compared with full fine-tuning, and both leave the base model frozen.

**Kubeflow-specific notes**

- All outputs in this notebook go under `~/lora_qlora_demo/`. That path is on the PVC, so adapters survive pod restarts.
- The HF model cache is set to `~/.cache/huggingface/`, also on the PVC — models you download once stay downloaded.
- If you stop / restart the pod and packages get wiped from `/opt/conda`, re-run `bash ~/setup_env.sh` and restart the kernel. Adapters and downloaded models stay.

**Suggested extension exercises**

1. Increase the dataset size and observe loss curves.
2. Change LoRA rank from 8 to 16 (or 4) and compare trainable params + final loss.
3. Try a different prompt format.
4. Compare GPU memory usage between LoRA and QLoRA using Section 17 before and after model load.
5. Push the saved adapters to Hugging Face Hub with `huggingface-cli login` + `model.push_to_hub(repo_id)`.